In [1]:
%reload_ext autoreload
%autoreload 2


In [2]:

import os
#os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.98"

import jax
import jax.numpy as jnp

import numpy as np
import matplotlib.pyplot as plt
from dmri.simulators import  Ball3StickSharedDiffusivity
from dmri.simulators.acquisition_scheme import acquisition_scheme
from flax import nnx

import matplotlib.pyplot as plt

plt.style.use("../dmri/utils/pyloric.mplstyle")


In [3]:
from dmri.train.utils import load_cfg, load_checkpoint

#checkpoint, model, simulator = load_checkpoint("/home/macke/mgloeckler90/dmri/results/new_multishell_ball3stick_simformer6_v_pred")
checkpoint, model, simulator = load_checkpoint("/home/macke/mgloeckler90/dmri/results/updated_all_3_6_8_128_model_prior")
graphdef, params,  state = nnx.split(model, nnx.Param,  ...)
params = checkpoint["params_ema"]

model = nnx.merge(graphdef, params, state)
model.eval()

In [4]:
sim_type = model.tokenizer.simulator
dataset_synthetic_short = jax.vmap(simulator[0])(jax.random.split(jax.random.PRNGKey(1), num=1000))
dataset_synthetic_long = jax.vmap(simulator[1])(jax.random.split(jax.random.PRNGKey(2), num=1000))

In [6]:
from dipy.data import get_fnames
from dipy.io.gradients import read_bvals_bvecs
from dipy.io.image import load_nifti

import numpy as np

folder_name = "data"
def load_real_data(folder_name):
    data, affine = load_nifti(f"../data/{folder_name}/data.nii.gz")
    brain_mask, _ = load_nifti(f"../data/{folder_name}/nodif_brain_mask.nii.gz")
    _bvals, bvecs = read_bvals_bvecs(f"../data/{folder_name}/bvals", f"../data/{folder_name}/bvecs")

    # Round bvals to nearest (0, 1000, 2000)
    bvals_rounded = np.round(_bvals / 1000) * 1000
    bvals = np.clip(bvals_rounded, 0, None)
    b0_mask = bvals == 0
    print(data.shape)

    S0 = np.mean(data[..., b0_mask], -1)

    data_norm = data / S0[..., None]
    data_norm = np.where(brain_mask[...,None], data_norm, 0.)
    data_norm = np.where(np.isnan(data_norm), 0., data_norm)
    S0 = np.where(brain_mask, S0, 0.)


    idx = np.argsort(bvals)

    bvals = bvals[idx]
    bvecs = bvecs[idx]
    data_norm = data_norm[...,idx]

    full_data_flat = data_norm.reshape(-1, data_norm.shape[-1])
    brain_mask_flat = brain_mask.reshape(-1)

    full_data_flat_in_brain = full_data_flat[brain_mask_flat.astype(np.bool),:]
    full_data_flat_in_brain = np.nan_to_num(full_data_flat_in_brain, nan=0, posinf=0, neginf=0)

    exclude_outliers = np.quantile(full_data_flat_in_brain, 0.999)
    full_data_flat_in_brain = np.clip(full_data_flat_in_brain, 0, exclude_outliers)

    acq = acquisition_scheme(np.array(bvals), np.array(bvecs))

    return {'x': full_data_flat_in_brain, 'acq': acq}
data_real = load_real_data(folder_name)

(104, 104, 72, 105)


/tmp/ipykernel_3895215/202660545.py:21: RuntimeWarning: divide by zero encountered in divide
  data_norm = data / S0[..., None]
/tmp/ipykernel_3895215/202660545.py:21: RuntimeWarning: invalid value encountered in divide
  data_norm = data / S0[..., None]


In [11]:
acq = data_real['acq']
x_single = data_real['x'][150_000]

In [15]:
acq

acquisition_scheme(bvals=array([   0.,    0.,    0.,    0.,    0., 1000., 1000., 1000., 1000.,
       1000., 1000., 1000., 1000., 1000., 1000., 1000., 1000., 1000.,
       1000., 1000., 1000., 1000., 1000., 1000., 1000., 1000., 1000.,
       1000., 1000., 1000., 1000., 1000., 1000., 1000., 1000., 1000.,
       1000., 1000., 1000., 1000., 1000., 1000., 1000., 1000., 1000.,
       1000., 1000., 1000., 1000., 1000., 1000., 1000., 1000., 1000.,
       1000., 2000., 2000., 2000., 2000., 2000., 2000., 2000., 2000.,
       2000., 2000., 2000., 2000., 2000., 2000., 2000., 2000., 2000.,
       2000., 2000., 2000., 2000., 2000., 2000., 2000., 2000., 2000.,
       2000., 2000., 2000., 2000., 2000., 2000., 2000., 2000., 2000.,
       2000., 2000., 2000., 2000., 2000., 2000., 2000., 2000., 2000.,
       2000., 2000., 2000., 2000., 2000., 2000.]), bvecs=array([[ 5.67198e-01,  4.42622e-01, -6.94530e-01],
       [-5.67198e-01, -4.42622e-01,  6.94530e-01],
       [-5.67198e-01, -4.42622e-01,  6.94530e-

In [10]:
b3s_mask = jnp.array([True, True, True, True] + [False] * 6 + [True, False])
b3t_mask = jnp.array([True] + [False] * 6 + [True] * 3 + [True, False])

In [21]:
log_prob_b3s = model.log_prob_mask(b3s_mask, acq, x_single, mask_prior=1.0)
log_prob_b3t = model.log_prob_mask(b3t_mask, acq, x_single, mask_prior=1.0)

In [29]:
log_prob_b3s, log_prob_b3t

(Array(-11.467609, dtype=float32), Array(-8.005676, dtype=float32))

In [61]:
evidence_b3s = model.estimate_evidence(jax.random.key(0), acq, x_single, b3s_mask, num_samples=1024, batch_size=128)

In [ ]:
evidence_b3t = model.estimate_evidence(jax.random.key(0), acq, x_single, b3t_mask, num_samples=1024, batch_size=128)

In [63]:
evidence_b3s, evidence_b3t

((Array(167.7204, dtype=float32), Array(0.06308644, dtype=float32)),
 (Array(169.86551, dtype=float32), Array(0.05385334, dtype=float32)))

In [28]:
# Evidence can be split into mean predictive log-likelihood and KL divergence to the prior. Let's look at these separately.

In [33]:
def sample_thetas_b3s(key):
    return model.sample_theta(key, acq, x_single, b3s_mask)
def sample_thetas_b3t(key):
    return model.sample_theta(key, acq, x_single, b3t_mask)

thetas_b3s = jax.vmap(sample_thetas_b3s)(jax.random.split(jax.random.key(0), num=10000))
thetas_b3t = jax.vmap(sample_thetas_b3t)(jax.random.split(jax.random.key(0), num=10000))

In [66]:
# Correct samples
def sir(thetas, model_mask):
    def logp(theta):
        sim = sim_type.from_theta(theta, model_mask=model_mask)
        return sim.log_likelihood(acq, x_single)
    def logq(theta):
        return model.log_prob_theta(theta, acq, x_single, model_mask)
    logq = jax.lax.map(logq, thetas, batch_size=1000)
    logp = jax.lax.map(logp, thetas, batch_size=1000)
    logweights = logp - logq
    weights = jax.nn.softmax(logweights - jax.scipy.special.logsumexp(logweights))
    return weights
weights_b3s = sir(thetas_b3s, b3s_mask)
weights_b3t = sir(thetas_b3t, b3t_mask)

In [74]:
thetas_b3s_resampled = jax.random.choice(jax.random.key(1), thetas_b3s, shape=(2000,), p=weights_b3s)
thetas_b3t_resampled = jax.random.choice(jax.random.key(1), thetas_b3t, shape=(2000,), p=weights_b3t)

In [93]:
jnp.unique(thetas_b3s_resampled, axis=0).shape, jnp.unique(thetas_b3t_resampled, axis=0).shape

((396, 51), (101, 51))

In [77]:
def pred_loglikelihood_b3s(thetas):
    def loglikelihood(theta):
        sim = sim_type.from_theta(theta, model_mask=b3s_mask)
        return sim.log_likelihood(acq, x_single)
    return jax.vmap(loglikelihood)(thetas)

def pred_loglikelihood_b3t(thetas):
    def loglikelihood(theta):
        sim = sim_type.from_theta(theta, model_mask=b3t_mask)
        return sim.log_likelihood(acq, x_single)
    return jax.vmap(loglikelihood)(thetas)

pred_loglikelihoods_b3s = pred_loglikelihood_b3s(thetas_b3s_resampled).mean()
pred_loglikelihoods_b3t = pred_loglikelihood_b3t(thetas_b3t_resampled).mean()

In [81]:
pred_loglikelihoods_b3s, pred_loglikelihoods_b3t

(Array(191.70193, dtype=float32), Array(196.43338, dtype=float32))

In [89]:
def kl_post_vs_prior_from_particles(thetas, weights, model_mask, logZ):
    weights = weights / jnp.sum(weights)

    theta_mask = sim_type.theta_mask(model_mask=model_mask)

    def loglik(theta):
        sim = sim_type.from_theta(theta, model_mask=model_mask)
        return sim.log_likelihood(acq, x_single)

    logliks = jax.lax.map(loglik, thetas)  # or jax.vmap(loglik)(thetas)

    # KL(p(θ|x) || p(θ)) = E_post[loglik] - logZ
    return jnp.sum(weights * logliks) - logZ

In [92]:
kl_post_vs_prior_from_particles(thetas_b3s, weights_b3s, b3s_mask, evidence_b3s)

KeyboardInterrupt: 

In [85]:
kl_reg_b3t = kl_reg_b3t(thetas_b3t)

In [88]:
kl_reg_b3s, kl_reg_b3t

(Array(191.63794, dtype=float32), Array(196.42226, dtype=float32))

In [72]:
pred_loglikelihoods_b3s - kl_reg_b3s

Array(171.27913, dtype=float32)

In [73]:
pred_loglikelihoods_b3t - kl_reg_b3t

Array(176.3971, dtype=float32)